In [1]:
import os
import numpy as np
from tqdm import tqdm
from scipy.spatial.distance import cdist

SKELETON_PATH = "skeletons_yolo/001"
X_train = []
y_train = []


In [2]:

def load_all_people_sequence(folder_path):
    files = sorted([f for f in os.listdir(folder_path) if f.endswith('.npy')])
    sequence = []
    for f in files:
        arr = np.load(os.path.join(folder_path, f))
        if arr.shape[0] == 0:
            sequence.append([])
        else:
            sequence.append(arr)
    return sequence, files


In [3]:
def get_label_from_filename1(fname):
    num = int(fname.split("_")[1].split(".")[0])
    if 0 <= num <= 37:
        return 0  # walking
    elif 38 <= num <= 49:
        return 1  # suspicious
    elif 50 <= num <= 57:
        return 2  # running
    return None

In [4]:
def track_people_across_frames(frame_landmarks):
    tracks = {}
    next_id = 0
    previous_centroids = []
    previous_ids = []

    for people in frame_landmarks:
        # výpočty centroidů
        current_centroids = [
            np.nanmean(p, axis=0)
            for p in people
            if isinstance(p, np.ndarray) and p.shape == (17, 2) and not np.isnan(p).all()
        ]

        # pokud není co porovnávat, tak skip tento frame
        if len(current_centroids) == 0 or len(previous_centroids) == 0:
            previous_centroids = current_centroids
            previous_ids = [-1] * len(current_centroids)
            continue

        assignments = [-1] * len(current_centroids)
        dist_matrix = cdist(current_centroids, previous_centroids)

        for i, row in enumerate(dist_matrix):
            j = np.argmin(row)
            if row[j] < 0.1:
                assignments[i] = previous_ids[j]

        for i, track_id in enumerate(assignments):
            if track_id == -1:
                track_id = next_id
                next_id += 1
            if track_id not in tracks:
                tracks[track_id] = []
            tracks[track_id].append(people[i])

        previous_centroids = current_centroids
        previous_ids = assignments

    return tracks

In [5]:
for fname in sorted(os.listdir(SKELETON_PATH)):
    if not fname.endswith(".npy"):
        continue

    arr = np.load(os.path.join(SKELETON_PATH, fname))
    label = get_label_from_filename1(fname)

    for person in arr:
        if person.shape != (17, 2) or np.isnan(person).all():
            continue
        X_train.append(person.flatten())  # (34,)
        y_train.append(label)

X_train = np.array(X_train)
y_train = np.array(y_train)

np.save("X_train_flat.npy", X_train)
np.save("y_train_flat.npy", y_train)

print("✅ Done")
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

✅ Done
X_train shape: (735, 34)
y_train shape: (735,)


In [6]:
X_train

array([[641.4732  , 212.08661 , 645.23346 , ..., 328.45892 , 620.3961  ,
        298.54858 ],
       [836.97345 , 186.0247  , 836.6809  , ..., 282.2897  , 790.4629  ,
        282.0243  ],
       [593.39716 , 174.65356 , 592.61926 , ..., 266.1806  , 553.3669  ,
        261.93637 ],
       ...,
       [830.48926 , 100.05579 , 829.1906  , ..., 188.54384 , 785.59454 ,
        199.41098 ],
       [853.1335  ,  75.09751 , 853.50604 , ..., 168.54474 , 814.45605 ,
        170.15154 ],
       [713.9065  ,  61.294697, 713.8615  , ..., 117.474236, 655.8643  ,
        118.85134 ]], dtype=float32)

In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import load_model
import numpy as np

X = np.load("X_train_flat.npy")
y = np.load("y_train_flat.npy")

y_cat = to_categorical(y)

def create_model():
    model = Sequential([
    Dense(256, activation='relu', input_shape=(34,)),
    Dropout(0.2),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(3, activation='softmax')
])

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.summary()

    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', patience=5, factor=0.5, verbose=1)

    model.fit(X, y_cat,
              epochs=1000,
              batch_size=32,
              validation_split=0.2,
              callbacks=[early_stop, reduce_lr])

    model.save("frame_classifier_mlp.keras")


In [16]:
model = load_model("frame_classifier_mlp.keras")
loss, acc = model.evaluate(X, y_cat)
print(f"Test accuracy: {acc:.2%}")


23/23 [==============================] - 0s 2ms/step - loss: 25.0613 - accuracy: 0.6748
Test accuracy: 67.48%


In [11]:
import numpy as np
y = np.load("y_train_flat.npy")
np.bincount(y)



array([496, 158,  81], dtype=int64)

In [12]:
from sklearn.metrics import classification_report
y_pred = np.argmax(model.predict(X), axis=1)
print(classification_report(y, y_pred))


23/23 [==============================] - 0s 909us/step
              precision    recall  f1-score   support

           0       0.67      1.00      0.81       496
           1       0.00      0.00      0.00       158
           2       0.00      0.00      0.00        81

    accuracy                           0.67       735
   macro avg       0.22      0.33      0.27       735
weighted avg       0.46      0.67      0.54       735



C:\Users\tomas\anaconda3\envs\tf210gpu\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\tomas\anaconda3\envs\tf210gpu\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\tomas\anaconda3\envs\tf210gpu\lib\site-packages\sklearn\metrics\_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [13]:
model.summary()

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_14 (Dense)            (None, 128)               4480      
                                                                 
 dropout_10 (Dropout)        (None, 128)               0         
                                                                 
 dense_15 (Dense)            (None, 64)                8256      
                                                                 
 dropout_11 (Dropout)        (None, 64)                0         
                                                                 
 dense_16 (Dense)            (None, 3)                 195       
                                                                 
Total params: 12,931
Trainable params: 12,931
Non-trainable params: 0
_________________________________________________________________
